# CICCADA standard plots

One section per plot family. The plotting code lives in `lib/`:

| File | Contents |
|---|---|
| `lib/palette.py` | Every colour and the global style (`use_style`). No other file holds a hex code. |
| `lib/sources.py` | Data loaders. All return the same tidy frame, so any plot works with any source. |
| `lib/a_site_day.py` | Family A: site-day time series |
| `lib/b_qv_response.py` | Family B: Q–V response |
| `lib/c_categories.py` | Family C: category breakdown |
| `lib/d_rates.py` | Family D: conformance rate by group or over time |
| `lib/e_slopes.py` | Family E: response-slope distributions |
| `lib/f_curtailment.py` | Family F: curtailment concentration |

In [ ]:
%load_ext autoreload
%autoreload 2

from lib import palette, sources
from lib import a_site_day, b_qv_response, c_categories, d_rates, e_slopes, f_curtailment

palette.use_style("notebook")

## Data source

| `SOURCE` | Where from | Needs |
|---|---|---|
| `"synthetic"` | `synthetic_site_day.csv` in this folder (invented data, regenerate with `sources.make_synthetic()`) | nothing |
| `"local"` | Local copy of Solar Analytics data: `ami_raw_phaseseparate` (PV circuits, polarity-corrected) and `ami_site_metadata` in `%LOCALAPPDATA%/ciccada/ami_store`, extracted from Athena for the synthetic-AMI work | the site and month are in that extract (Jan 2024 – Dec 2025, ~10,500 sites) |
| `"aws"` | Athena, `solar_analytics_iceberg.ts` | `aws sso login --profile ciccada` |

Every loader returns `(df, info)`: `df` has `t_aest, phase, V, P_kw, Q_kvar, P_pot_kw`; `info` has `rating_kva`, `label`, `date`, `source` and `site` (metadata from `meta_up23c` / `ami_site_metadata`: state, postcode, DNSP, manufacturer, model, inverter count, AC/DC capacity, export limit, install and monitoring dates, flexible export, plus voltage class, monitor type and polarity of the PV circuits). Each figure cell prints the site metadata and day statistics above the figure (they are not drawn on it); pass `show_meta=False` to skip them.

In [ ]:
SOURCE = "local"          # "synthetic" | "local" | "aws"

# Real-data examples, used when SOURCE is "local" or "aws". 
# DATE = None -> best day of MONTH (sources.list_days).
# Candidate sites: see the table below, or run the example finder below.
SITE_1P, MONTH_1P, DATE_1P = 1808312180, "2025-01", "2025-01-20"
SITE_3P, MONTH_3P, DATE_3P = 1327502909, "2025-01", "2025-01-24"


def get(example):
    if SOURCE == "synthetic":
        return sources.load_synthetic(example)
    site, month, date = (SITE_1P, MONTH_1P, DATE_1P) if example == "1P" else (SITE_3P, MONTH_3P, DATE_3P)
    if site is None:
        return None, None
    if date is None:                     # rank the month's days, take the top one
        days = sources.list_days(site, month, SOURCE)
        display(days)
        date = days["date"].iloc[0]
    loader = {"local": sources.load_local, "aws": sources.load_aws}[SOURCE]
    return loader(site, date)


df_1p, info_1p = get("1P")
df_3p, info_3p = get("3P")

### Example sites

Found with `sources.find_examples` on the local store (Dec 2024 and Jan 2025), using the same point-wise rules the plots draw. A mode **conforms** on a day if ≤ 10 % of its judged intervals are outside the requirement, and **fails** if ≥ 50 %. Volt-VAr is judged on assessable intervals above 240 V; Volt-Watt on intervals above 253 V with P ≥ 20 % of rating. Every pick except 1598853781 also agrees with that site-month's slope-method verdict from the PV circuits (notebook 13).

| Cohort | Volt-VAr | Volt-Watt | Site | Day | Rating (kW) | VV outside band | VW above ceiling |
|---|---|---|---|---|---|---|---|
| 1P | conforms | conforms | 1808312180 | 2024-12-25 | 4.99 | 3 % of 133 | 0 % of 109 |
| 1P | conforms | conforms | 993464143 | 2024-12-10 | 10 | 0 % of 130 | 0 % of 88 |
| 1P | conforms | *closest to failing* | 873861233 | 2024-12-17 | 9 | 2 % of 106 | 32 % of 31 |
| 1P | fails | conforms | 1295784779 | 2024-12-23 | 10 | 100 % of 119 | 0 % of 100 |
| 1P | fails | conforms | 2066603468 | 2024-12-27 | 10 | 100 % of 130 | 0 % of 88 |
| 1P | fails | fails | 465008538 | 2025-01-17 | 8 | 100 % of 114 | 74 % of 108 |
| 1P | fails | fails | 798827521 | 2024-12-12 | 8 | 100 % of 140 | 71 % of 106 |
| 3P | conforms | conforms | 1327502909 | 2025-01-24 | 29.9 | 7 % of 123 | 0 % of 35 |
| 3P | fails | conforms | 741657766 | 2025-01-13 | 30 | 100 % of 112 | 0 % of 78 |
| 3P | fails | conforms | 530210663 | 2024-12-29 | 12 | 100 % of 122 | 0 % of 40 |
| 3P | fails | fails | 1598853781 | 2025-01-25 | 29.9 | 100 % of 129 | 75 % of 88 |
| 3P | fails | fails | 2145720865 | 2025-01-18 | 10 | 100 % of 96 | 50 % of 8 |

No day in either month both conforms with Volt-VAr and fails Volt-Watt, for 1P or 3P; 873861233 is the closest. 1598853781 has three PV circuits but is not in the notebook-13 fleet, so it has no slope verdict.

To search other months: `sources.find_examples("YYYY-MM")` (about 30 s per month; `all_days=True` returns every site-day).

In [ ]:
FIND_EXAMPLES = False          # set True to scan a month (a few minutes)

if FIND_EXAMPLES:
    examples = sources.find_examples("2025-01", n=3)
    display(examples)

---
## A. Site-day time series

**Shape:** 

Stacked panels sharing a 00:00 to 24:00 AEST axis (fixed UTC+10), always in this order: 
voltage → active power → (Volt-Watt NC) → reactive power → (Volt-VAr NC). 

Fixed y ranges so days are comparable: 
- V = [215, 265]
- P −5 to 115 %
- Q −70 to +55 % (widened only if the data go outside). 

P and Q are % of rating on the left, kW / kvar on the right (same scale, relabelled). Night (no generation) is shaded light grey.

**Three-phase:** 
- One thin line per phase (L1 red, L2 light grey for "white", L3 dark blue) plus one thick line in the quantity colour. 
- V: the thick dashed line is the average of the phases (the reference voltage for a three-phase inverter, AS/NZS 4777.2 Cl. 3.3.2.1). 
- P and Q: the thick line is the sum of the phases
- phases and sum are all % of the site rating. 
- Conformance is assessed on the thick lines.

**Assessment:** 

uses `bms_sa_review/shared/as4777_curves.py`, the same functions as the Stage 2 conformance tables (`review_corrected`):
- Volt-VAr band = required curve ± 4 % of rating, capped by Figure 2.1: none below 20 % P (not assessable, hatched grey, excluded from totals), 44 % for 20–60 %, PF 0.8 for 60–80 %, no relaxation above 80 %.
- Volt-Watt ceiling = curve + 4 % of rating, assessed only when V > 253 V.

### A.0 Operational modes: Volt-Watt and Volt-VAr

In [ ]:
fig = a_site_day.plot_operational_day(df_1p, info_1p)

In [ ]:
if df_3p is not None:
    fig = a_site_day.plot_operational_day(df_3p, info_3p)

### A.1 Volt-VAr response
The P panel shows the 20 % gate. In the Q panel the dashed band is what is required after the Figure 2.1 cap; the dotted line is the uncapped Volt-VAr curve.

In [ ]:
fig = a_site_day.plot_voltvar_day(df_1p, info_1p)

In [ ]:
if df_3p is not None:
    fig = a_site_day.plot_voltvar_day(df_3p, info_3p)

### A.2 Volt-Watt response
Shaded where V > 253 V (Volt-Watt active). Volt-Watt is a cap, so any P at or below the ceiling is permitted: the dotted line is the Volt-Watt curve, the grey band the 4 % tolerance above it, and its dashed top edge the ceiling that is assessed. If the data carry `P_pot_kw` (PV potential / counterfactual), the energy curtailed while exposed is filled orange.

In [ ]:
fig = a_site_day.plot_voltwatt_day(df_1p, info_1p)

In [ ]:
if df_3p is not None:
    fig = a_site_day.plot_voltwatt_day(df_3p, info_3p)

### A.3 Protective functions (sustained operation, anti-islanding)
**TO BE DONE.** `a_site_day.plot_protective_day` is a placeholder.

---
## B. Q–V response

**Shape:** reactive power against voltage. x = voltage, 205–265 V; y = Q as % of rating (+ supplying, − absorbing), −70 to +55 %. Behind the data: the Volt-VAr curve (black dashed) with its ±4 % tolerance band, the set-point voltages (207, 220, 240, 258 V, dotted) and the slope-method window 242–253 V (light grey).

**Points:** only assessable intervals (|P| ≥ 20 % of rating) count towards medians and fits; the rest are drawn as faint hollow grey circles.

**Three-phase:** mean of the phase voltages against the sum of the phase Q, as in family A and the 3L3P method.

**Slope:** OLS of Q on V over 242–253 V (≥ 20 assessable points spanning ≥ 5 V). Required slope −3.33 %/V; in band if within ±20 % (−4.00 to −2.67 %/V), as in notebooks 05–13.

In [ ]:
# Several days of one site (B.1, B.2): the whole month of the family-A examples
def get_month(example):
    if SOURCE == "synthetic":
        return sources.load_synthetic_month(example)
    site, month = (SITE_1P, MONTH_1P) if example == "1P" else (SITE_3P, MONTH_3P)
    if site is None:
        return None, None
    return sources.load_month(site, month, SOURCE)


dfm_1p, infom_1p = get_month("1P")
dfm_3p, infom_3p = get_month("3P")

### B.1 Every interval
One point per interval, the median per 1 V bin (blue with a black edge) and the slope fitted over 242–253 V (purple dashed; verdict in the legend). The box under the figure gives the fit, the assessable intervals inside and outside the permitted band (Figure 2.1 capped, as in family A), and the outside-band intervals per category (adverse, inactive, significant shortfall, near-conformant, major surplus; Stage 2 Q_impact thresholds). `colour_by="category"`, `"hour"` or `"P"` colours the assessable points by category, time of day or active power.

In [ ]:
fig = b_qv_response.plot_qv_scatter(dfm_1p, infom_1p)

In [ ]:
if dfm_3p is not None:
    fig = b_qv_response.plot_qv_scatter(dfm_3p, infom_3p)

### B.2 Day by day
One thin median curve per day (coloured by date) over the month's median and 25–75 % range: is the response the same every day?

In [ ]:
fig = b_qv_response.plot_qv_curve(dfm_1p, infom_1p)

In [ ]:
if dfm_3p is not None:
    fig = b_qv_response.plot_qv_curve(dfm_3p, infom_3p)

### B.3 Across sites
Each site is reduced to its median Q per 1 V bin (assessable intervals) with `sources.qv_profiles(month)`, about 30 s per month from the local store. The figure shows, per bin, the median across sites and the 25–75 % and 10–90 % ranges; the strip below counts sites per bin (blue = bins drawn). Here the sites are split by their slope-method verdict for the same month (`sources.slope_verdicts`, notebook 13). Not available for `SOURCE = "aws"`.

In [ ]:
FLEET_MONTH = "2025-01"

if SOURCE == "synthetic":
    profiles, info_fleet = sources.load_synthetic_fleet()
    profiles["verdict"] = profiles["conformant"].map({True: "pass", False: "fail"})
elif SOURCE == "local":
    profiles = sources.qv_profiles(FLEET_MONTH)
    verdicts = sources.slope_verdicts(FLEET_MONTH)
    profiles = profiles.merge(verdicts[["site_id", "pass_truth"]], on="site_id", how="inner")
    profiles["verdict"] = profiles["pass_truth"].map({True: "pass", False: "fail"})
    info_fleet = {"label": "Solar Analytics PV circuits (local store)", "date": FLEET_MONTH}
else:
    profiles = None
    print("Fleet profiles need the local store; set SOURCE to 'local' or 'synthetic'.")

In [ ]:
if profiles is not None:
    for cohort in ["1P", "3P"]:
        sub = profiles[profiles["cohort"] == cohort]
        if sub.empty:
            continue
        fig = b_qv_response.plot_qv_fleet(
            sub, {**info_fleet, "label": f"{info_fleet['label']}  ·  {cohort}"},
            group="verdict", group_order=["pass", "fail"],
            group_labels={"pass": "Slope method: pass", "fail": "Slope method: fail"})

---
## C. Category breakdown

**Shape:** 100 % stacked bars, categories always in the same order, worst → best → not judged, with the legend in that order. The number of intervals or sites behind each bar is printed at its end; shares of at least 6 % are written on the bar.

**Categories** (same rules as families A and B, computed per 5-min interval on the site total):
- **Volt-VAr:** adverse, inactive, significant shortfall, near-conformant (outside the Figure-2.1-capped band, split by Q_impact as in the Stage 2 tables), conformant (inside the band, outside the 220–240 V deadband), major surplus. Not judged: not exposed (inside the band, in the deadband) and not assessable (|P| < 20 % of rating).
- **Volt-Watt:** above the ceiling, at or below the ceiling. Not judged: not exposed (V ≤ 253 V).

**Units:** `unit="intervals"` counts 5-min intervals; `unit="sites"` gives one verdict per site: conformant if ≤ 10 % of its judged intervals are non-conforming (`SITE_CONF_THRESH`), not assessable if it has fewer than 12 judged intervals. By default every outside-band Volt-VAr category counts as non-conforming; `vv_nonconf=[...]` changes that.

**Groups:** `c_categories.add_groups` adds `install_era` (before / after 18 Dec 2021), `install_year`, `oem` and `dnsp` (largest 8, the rest "Other"; `anonymise=True` replaces the names with "Manufacturer 1", "Network 1" …).

In [ ]:
# Per-site category counts for one month (family C fleet figures)
if SOURCE == "synthetic":
    counts, info_counts = sources.load_synthetic_category_counts()
elif SOURCE == "local":
    counts = sources.category_counts(FLEET_MONTH)          # about a minute: reads every interval of the month
    info_counts = {"label": "Solar Analytics PV circuits (local store)", "date": FLEET_MONTH}
else:
    counts = None
    print("Category counts need the local store; set SOURCE to 'local' or 'synthetic'.")

if counts is not None:
    counts = c_categories.add_groups(counts)

### C.1 By group
Sorted with the largest non-conforming share at the top (`sort="worst"`); ordered groups such as install year use `sort=None`.

In [ ]:
if counts is not None:
    table = c_categories.category_table(counts, by="cohort", mode="voltvar", unit="intervals")
    fig = c_categories.plot_category_bars(table, mode="voltvar", unit="intervals", info=info_counts)

In [ ]:
if counts is not None:
    table = c_categories.category_table(counts, by="dnsp", mode="voltvar", unit="intervals")
    fig = c_categories.plot_category_bars(table, mode="voltvar", unit="intervals", info=info_counts)

In [ ]:
if counts is not None:
    table = c_categories.category_table(counts, by="install_year", mode="voltvar", unit="sites")
    fig = c_categories.plot_category_bars(table, mode="voltvar", unit="sites", info=info_counts, sort=None)

In [ ]:
if counts is not None:
    table = c_categories.category_table(counts, by="oem", mode="voltwatt", unit="intervals")
    fig = c_categories.plot_category_bars(table, mode="voltwatt", unit="intervals", info=info_counts)

### C.2 One site, day by day
The month of the family-B examples. Each bar is one day's judged intervals; the number above it is how many there were.

In [ ]:
fig = c_categories.plot_category_days(dfm_1p, infom_1p)

In [ ]:
if dfm_3p is not None:
    fig = c_categories.plot_category_days(dfm_3p, infom_3p)

### C.3 Both modes at once
Site verdicts on Volt-VAr and Volt-Watt together, for sites with at least 12 judged intervals on each mode.

In [ ]:
if counts is not None:
    fig = c_categories.plot_joint_bars(counts, by="install_era", info=info_counts)

---
## D. Conformance rate by group or over time

**Shape:** share of sites (or site-months) that conform, 0–100 %. Every bar or point has its n and an error bar (or band) showing the **95 % confidence interval** of the rate. Ordered groups (install year, month) run along x; unordered groups (network, manufacturer) are horizontal bars sorted by rate, with a dotted line at the overall rate. Install-year figures mark the date AS/NZS 4777.2:2020 became mandatory (18 Dec 2021). Group × year grids use the rate ramp (Blues), with cells below the minimum n hatched.

**The error bars.** A group's conformance rate comes from a finite number of sites, so it is an estimate. The 95 % confidence interval shows how precise it is: treating the group's sites as a sample, an interval built this way contains the underlying rate in about 95 of 100 samples. Narrow bars (large n) mean a precise rate; wide bars (small n) mean the rate could easily be quite different. It is not a percentile of the data. The interval is computed with the **Wilson score** method ([Wilson 1927](https://scilit.com/publications/51ab62bc5a19b66438acfe9c66479b31)) rather than the textbook p ± 1.96·√(p(1−p)/n): Wilson's interval never goes below 0 % or above 100 % and stays reliable for small groups and for rates close to 0 % or 100 %, where the textbook formula fails ([Brown, Cai & DasGupta 2001](https://www.doi.org/10.1214/SS/1009213286)). Caveat: the months of one site are not independent, so intervals on site-months are narrower than they should be; site-level intervals do not have this problem.

**Colours:** one verdict alone is drawn in the conformant blue. Net-meter and PV-circuit verdicts side by side use the data-source colours (net meter orange, PV circuits blue); other series use the group colour cycle.

**Input:** any table with one row per site or site-month, boolean verdict column(s) and the grouping columns. Here:
- slope method, notebook 13 (`sources.slope_verdicts()`): `passed` = net meter (blind slope), `pass_truth` = PV circuits (reference slope); network and manufacturer are the anonymised codes from notebook 13;
- site level with `d_rates.site_level(...)`: conformant only if the site passes in every assessable month (notebook 13 rule; `rule="half"` for ≥ 50 % of months);
- point-wise verdicts from family C (`c_categories.site_verdicts(counts)`).

In [ ]:
# Slope-method verdicts (notebook 13): one row per site-month, 2024-01 to 2025-12
if SOURCE == "synthetic":
    site_months, info_rates = sources.load_synthetic_slope_verdicts()
elif SOURCE == "local":
    site_months = sources.slope_verdicts()
    info_rates = {"label": "Solar Analytics, slope method (notebook 13)", "date": "2024-01 to 2025-12"}
else:
    site_months = None
    print("Slope verdicts are read from the local store; set SOURCE to 'local' or 'synthetic'.")

if site_months is not None:
    sites = d_rates.site_level(site_months, ok="passed", rule="all")

### D.1 By install year
Site-months, net meter and PV circuits side by side; then sites (net meter, conformant in every month).

In [ ]:
if site_months is not None:
    fig = d_rates.plot_rate_by_year(site_months, ok=["passed", "pass_truth"], info=info_rates, unit="site-months")

In [ ]:
if site_months is not None:
    fig = d_rates.plot_rate_by_year(sites, ok="conformant", info=info_rates, unit="sites")

### D.2 By group
Manufacturer (slope method, sites), then network from the point-wise verdicts of family C (`counts`, one month).

In [ ]:
if site_months is not None:
    fig = d_rates.plot_rate_by_group(sites, by="oem", ok="conformant", info=info_rates, unit="sites",
                                     group_label="manufacturer")

In [ ]:
if counts is not None:
    pointwise = c_categories.site_verdicts(counts)
    pointwise["vv_ok"] = (pointwise["vv_verdict"] == "conformant").where(pointwise["vv_verdict"] != "not_assessable")
    fig = d_rates.plot_rate_by_group(pointwise, by="dnsp", ok="vv_ok", info=info_counts, unit="sites",
                                     group_label="network (point-wise Volt-VAr, ≤ 10 % of judged intervals outside the band)")

### D.3 Over time
Site-months per calendar month, one line per cohort (net meter).

In [ ]:
if site_months is not None:
    fig = d_rates.plot_rate_over_time(site_months, ok="passed", series="cohort", info=info_rates)

### D.4 Group × install year
Network × install year, site-months (net meter): rate in each cell with (n) below it, on a fixed 0–100 % scale. Cells with fewer than 10 site-months are blank and hatched; the dashed line marks when AS/NZS 4777.2:2020 became mandatory (18 Dec 2021, so between install years 2021 and 2022).

In [ ]:
if site_months is not None:
    fig = d_rates.plot_rate_heatmap(site_months, row="dnsp", col="install_year", ok="passed", info=info_rates,
                                    unit="site-months", row_label="network", col_label="install year")

---
## E. Response-slope distributions

**Shape:** slope of Q (% of rating) on V over 242–253 V, in % of rating per volt, always on the same range (−8 to +3 %/V). The required slope (−3.33 %/V) is black dashed, the pass band (±20 %, −4.00 to −2.67 %/V) grey, 0 %/V (no response) a thin line. Every panel gives n and the share in the band; slopes outside the range are clipped to its edge.

**Colours:** one slope column in the Q blue; net meter and PV circuits together in the data-source colours (orange, blue); groups in the group colour cycle.

**Input:** the slope-method site-months from section D (`site_months`: `slope` = net meter, `slope_truth` = PV circuits).

### E.1 Histogram
Net meter and PV circuits overlaid, then one panel per cohort.

In [ ]:
if site_months is not None:
    fig = e_slopes.plot_slope_hist(site_months, cols=["slope", "slope_truth"], info=info_rates)

In [ ]:
if site_months is not None:
    fig = e_slopes.plot_slope_hist(site_months, cols="slope_truth", by="cohort", info=info_rates)

### E.2 By group
Box (median, 25–75 %, whiskers 5–95 %) and up to 300 points per group. The % above each box is the share in the band; the number below is n.

In [ ]:
if site_months is not None:
    fig = e_slopes.plot_slope_box(site_months, col="slope", by="install_year", info=info_rates,
                                  group_label="install year")

### E.3 Cumulative distribution
Share of site-months with a slope at or below x, one line per cohort. Where each line crosses the band edges gives the shares too steep, in band and too shallow.

In [ ]:
if site_months is not None:
    fig = e_slopes.plot_slope_ecdf(site_months, col="slope_truth", by="cohort", info=info_rates)

---
## F. Curtailment concentration

**Shape** (as Figure 4 of `oem_analysis/notebooks/99_publication_figures.ipynb`): F.1 puts a Lorenz curve per mechanism either side of a bar chart of total curtailment by mechanism (widths 40 / 20 / 40 %; with one mechanism only, bar chart then Lorenz). In each Lorenz panel, affected sites are ranked largest loss first: x = cumulative % of affected sites, y = cumulative % of total curtailment; the grey dashed diagonal is equal loss at every site; markers label the share held by the top 1, 5, 10 and 20 % of sites, the top 20 % is shaded, and the Gini coefficient is in the title. F.2 shows, per group, total curtailed energy (MWh), curtailed energy as a share of the group's generation, and the share of sites affected.

**Colours:** the publication colours, `palette.MECHANISM`: Volt-VAr-induced curtailment violet, Volt-Watt curtailment indigo.

**Input:** one row per site with `voltvar_kwh` / `voltwatt_kwh`, optionally `generation_kwh` and grouping columns. With `SOURCE = "local"` this is the **SolarEdge** fleet from the local OEM store (`sources.se_curtailment`, Volt-Watt only, GHI counterfactual, 2025): the Solar Analytics curtailment tables are only on Athena (`sources.sa_curtailment_aws`, untested), and the Volt-VAr-induced estimate needs a counterfactual that is not in the local store. The synthetic data have both mechanisms.

In [ ]:
if SOURCE == "synthetic":
    curtailment, info_curt = sources.load_synthetic_curtailment()
elif SOURCE == "local":
    curtailment = sources.se_curtailment(year=2025)
    info_curt = {"label": "SolarEdge fleet, Volt-Watt curtailment (GHI counterfactual, oem_analysis)", "date": "2025"}
else:
    curtailment = sources.sa_curtailment_aws(year=2025)
    info_curt = {"label": "Solar Analytics, Volt-Watt curtailment (Stage 2, GHI counterfactual)", "date": "2025"}

### F.1 Concentration

In [ ]:
fig = f_curtailment.plot_curtailment_summary(curtailment, info=info_curt)
f_curtailment.concentration(curtailment, mechanism="voltwatt")

### F.2 By group

In [ ]:
if "state" in curtailment:
    fig = f_curtailment.plot_energy_by_group(curtailment, by="state", info=info_curt)

## Day sources

In [ ]:
days = sources.find_examples("2025-01", all_days=True)
both = days[(days.volt_var == "conforms") & (days.volt_watt == "conforms")]
both.sort_values(["vw_n", "vv_n"], ascending=False).head(10)